# Flax: networks with explicit parameters

Section 15.4 of the notes.

A Flax module looks like a PyTorch module, but its parameters can be taken out
as a separate object, which is what the JAX transformations need.

In [ ]:
import jax
import jax.numpy as jnp
import optax
from flax import nnx


class MLP(nnx.Module):
    def __init__(self, din, dhidden, dout, rngs):
        self.l1 = nnx.Linear(din, dhidden, rngs=rngs)
        self.l2 = nnx.Linear(dhidden, dout, rngs=rngs)

    def __call__(self, x):
        return self.l2(jnp.tanh(self.l1(x)))


model = MLP(1, 32, 1, rngs=nnx.Rngs(0))
print("output shape:", model(jnp.ones((7, 1))).shape)

## 1. The parameters as data

`nnx.state` extracts the parameters. They form a tree of arrays, which `grad`
and `jit` accept as an argument.

In [ ]:
params = nnx.state(model, nnx.Param)
leaves = jax.tree.leaves(params)
print("arrays:", len(leaves), " numbers:", sum(leaf.size for leaf in leaves))
print("shapes:", [leaf.shape for leaf in leaves])

## 2. Fitting a function

The training loop is written with `nnx.value_and_grad` and an optimizer from
`optax`. The target is one period of a sine.

In [ ]:
key = jax.random.key(0)
X = jax.random.uniform(key, (256, 1), minval=-1.0, maxval=1.0)
Y = jnp.sin(jnp.pi * X)


def loss_fn(model):
    return jnp.mean((model(X) - Y) ** 2)


optimizer = nnx.Optimizer(model, optax.adam(1e-2), wrt=nnx.Param)


@nnx.jit
def train_step(model, optimizer):
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss


print("loss before:", float(loss_fn(model)))
for step in range(1, 501):
    loss = train_step(model, optimizer)
    if step % 100 == 0:
        print(f"  step {step:4d}  loss {float(loss):.5f}")
print("loss after: ", float(loss_fn(model)))

## 3. The fit

In [ ]:
import matplotlib.pyplot as plt

xs = jnp.linspace(-1, 1, 200).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(xs, jnp.sin(jnp.pi * xs), label="target")
ax.plot(xs, model(xs), "--", label="network")
ax.scatter(X, Y, s=4, alpha=0.3, label="training points")
ax.legend()
ax.set_xlabel("x")
plt.show()

## 4. The same network in PyTorch

For comparison, the PyTorch version keeps the parameters inside the module and
the optimizer reads their gradients.

In [ ]:
import torch
import torch.nn as nn

torch_net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))
opt = torch.optim.Adam(torch_net.parameters(), lr=1e-2)

Xt = torch.rand(256, 1) * 2 - 1
Yt = torch.sin(torch.pi * Xt)

for step in range(1, 501):
    opt.zero_grad()  # gradients accumulate, so clear them
    loss = ((torch_net(Xt) - Yt) ** 2).mean()
    loss.backward()
    opt.step()

print("final loss:", loss.item())